In [1]:
# Cài đặt theo hướng dẫn DeepSeek-OCR
!pip install -q torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers==4.46.3 tokenizers==0.20.3
!pip install -q PyMuPDF img2pdf einops easydict addict Pillow numpy pdf2image
!pip install -q flash-attn==2.7.3 --no-build-isolation
!apt-get install -qq -y poppler-utils

# Xóa cache
!rm -rf ~/.cache/huggingface/modules/transformers_modules

In [9]:
# Upload PDF
from google.colab import files
uploaded = files.upload()
pdf_file = list(uploaded.keys())[0]
print(f"Uploaded: {pdf_file}")

Saving test2.pdf to test2.pdf
Uploaded: test2.pdf


In [ ]:
# Cell 4: Xử lý tất cả PDF
from pdf2image import convert_from_path
import re, sys, zipfile
from io import StringIO

def process_pdf(pdf_path, model, tokenizer):
    images = convert_from_path(pdf_path, dpi=200)
    results = []

    for i, img in enumerate(images, 1):
        print(f'  Page {i}/{len(images)}...')
        img_path = f'temp_page_{i}.jpg'
        img.save(img_path)

        old_stdout = sys.stdout
        sys.stdout = captured = StringIO()

        model.infer(
            tokenizer,
            prompt='<image>\nConvert to markdown.',
            image_file=img_path,
            output_path='.',
            base_size=1024,
            image_size=640,
            crop_mode=True
        )

        sys.stdout = old_stdout
        output = captured.getvalue()

        clean = re.sub(r'<\|ref\|>.*?<\|/ref\|>', '', output)
        clean = re.sub(r'<\|det\|>.*?<\|/det\|>', '', clean)
        clean = re.sub(r'=+|BASE:.*|PATCHES:.*|image:.*|other:.*', '', clean)
        clean = clean.strip()

    return ''.join(results)

# Xử lý từng file
for idx, pdf_file in enumerate(pdf_files, 1):
    print(f'[{idx}/{len(pdf_files)}] Processing {pdf_file}...')
    markdown = process_pdf(pdf_file, model, tokenizer)
    output_name = pdf_file.replace('.pdf', '.md')

    with open(output_name, 'w', encoding='utf-8') as f:
        f.write(markdown)
    print(f'  ✓ Saved to {output_name}')

A new version of the following files was downloaded from https://huggingface.co/deepseek-ai/DeepSeek-OCR:
- configuration_deepseek_v2.py
- deepencoder.py
- conversation.py
- modeling_deepseekv2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.
Some weights of DeepseekOCRForCausalLM were not initialized from the model checkpoint at deepseek-ai/DeepSeek-OCR and are newly initialized: ['model.vision_model.embeddings.position_ids']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag 

Processing page 1/2...


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Processing page 2/2...


In [ ]:
# Download kết quả
from google.colab import files
files.download('output.md')